In [56]:
import pandas as pd 
import numpy as np 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
from tqdm import tqdm 
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


In [19]:
def read_data(data_path: str):
    """This function is used for read data from csv"""
    df = pd.read_csv(data_path)
    
    return df

def filter_ibft_data(df: pd.DataFrame):
    """This function is used for filtering value in column"""
    selected_columns = [
                        "group_id", "id", "requester_id", "cf_chi_tit_vn",
                        "subject", "description_text", "cf_longdtn_test",
                        "cf_i_tc145665", "expected_response", "conversations_clean", "conversations"
                   ]
    selected_df = df[selected_columns]
    selected_df =  selected_df[selected_df["cf_i_tc145665"] == "241 - Chuyển Tiền ATM"]
    selected_df =  selected_df.dropna()
    selected_df =  selected_df.reset_index(drop=True)

    return selected_df

In [48]:
def remove_transaction_info(text: str) -> str:
    pattern = (
        r"\(\s*Mã giao dịch:\s*[^)]+\)\s*-\s*\(\s*Ticket\s*id:\s*[^)]+\)"
        r"|\(\s*Ticket\s*id:\s*[^)]+\)"
        r"|\b\d{15}\b"  # Mã giao dịch dạng số 15 chữ số
    )

    text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    return re.sub(r"\s{2,}", " ", text).strip()

def parse_description_text(text: str) -> dict:
    result = {}

    # bỏ dấu + đầu chuỗi
    text = text.strip().lstrip("+")

    # tách từng field
    fields = re.split(r"\s+\+\s+", text)

    for field in fields:
        field = field.strip()

        if ":" not in field:
            continue

        key, value = field.split(":", 1)

        result[key.strip()] = value.strip()

    return result

def extract_problem(text: str) -> dict:
    """this function is used for summary problem of user"""
    description_json = parse_description_text(text)
    full_text =  ""
    selected_keys = [
                        "Mục đích chuyển tiền", 
                        "Có liên hệ người nhận chưa", "Mô tả"
                    ]
    for key, value in des.items():
        if key in selected_keys:
            full_text += f"\n{key}: {value}"
    full_text =  full_text.strip()  

    return full_text
    
def parser_data(df: pd.DataFrame):
    """This function is used for parser dataframe"""
    df["problem"] =  df["description_text"].apply(lambda x: extract_problem(x))
    df["clean_subject"] =  df["subject"].apply(lambda x: remove_transaction_info(x))
    df["classify_context"] =  "Tiêu đề: " + df["clean_subject"] + "\n" + df["problem"]
    
    return df    
    

In [49]:
data_path = "tickets_20260604_150859_with_labels.csv"
df =  read_data(data_path)
df_ibft =  filter_ibft_data(df)
df_ibft =  parser_data(df_ibft)

/tmp/ipykernel_6884/585209666.py:3: DtypeWarning: Columns (21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


In [51]:
text = df_ibft.iloc[0]['classify_context']
print(text, flush=True)

Tiêu đề: Trạng thái chuyển tiền "Thành công" nhưng tài khoản ngân hàng người nhận chưa được cộng tiền
Mục đích chuyển tiền: Bị trừ tiền ngang trông khi tôi không có chuyển tiền
Có liên hệ người nhận chưa: Không liên lạc được
Mô tả: Tiền trong ngân hàng mbbank tự rút về Zalopay xong tự chuyển khoảng??


In [53]:
model = SentenceTransformer("google/gemma-3-270m")

Some weights of Gemma3TextModel were not initialized from the model checkpoint at google/gemma-3-270m and are newly initialized: ['embed_tokens.weight', 'layers.0.input_layernorm.weight', 'layers.0.mlp.down_proj.weight', 'layers.0.mlp.gate_proj.weight', 'layers.0.mlp.up_proj.weight', 'layers.0.post_attention_layernorm.weight', 'layers.0.post_feedforward_layernorm.weight', 'layers.0.pre_feedforward_layernorm.weight', 'layers.0.self_attn.k_norm.weight', 'layers.0.self_attn.k_proj.weight', 'layers.0.self_attn.o_proj.weight', 'layers.0.self_attn.q_norm.weight', 'layers.0.self_attn.q_proj.weight', 'layers.0.self_attn.v_proj.weight', 'layers.1.input_layernorm.weight', 'layers.1.mlp.down_proj.weight', 'layers.1.mlp.gate_proj.weight', 'layers.1.mlp.up_proj.weight', 'layers.1.post_attention_layernorm.weight', 'layers.1.post_feedforward_layernorm.weight', 'layers.1.pre_feedforward_layernorm.weight', 'layers.1.self_attn.k_norm.weight', 'layers.1.self_attn.k_proj.weight', 'layers.1.self_attn.o_pro

In [57]:
existing_embeddings = []
selected_texts = []
selected_index = []
for index, row in df_ibft.iterrows():

    classify_context = row["classify_context"]
    emb = model.encode(classify_context, normalize_embeddings=True)

    if len(existing_embeddings) == 0:
        existing_embeddings.append(emb)
        selected_texts.append(classify_context)
        selected_index.append(index)
        continue

    sims = cosine_similarity(
        [emb],
        np.array(existing_embeddings)
    )[0]

    max_sim = sims.max()

    if max_sim < 0.85:
        existing_embeddings.append(emb)
        selected_texts.append(classify_context)
        selected_index.append(index)

print(f"Selected {len(selected_texts)} samples")

Selected 197 samples


In [ ]:
df_ibft.iloc[selected_index]